# Benchmark M1 · 20.000 bài arXiv trên Colab CPU

Mỗi thành viên mở **cùng notebook**, chọn CPU rồi bấm **Runtime → Run all**. Màn hình sẽ in lô vừa nhận và số ID đã ghi. Khi Colab thu hồi runtime, mở lại notebook và bấm **Run all**. Một thành viên khác cũng có thể tiếp tục.

| Chuẩn bị trong Colab | Thực hiện |
|---|---|
| **GITHUB_TOKEN** · bắt buộc | Trong biểu tượng chìa khóa **Secrets**, thêm token GitHub của chính bạn, bật **Notebook access**. Tài khoản cần quyền push vào nhánh kết quả của QuangPhat101/arXiv-Report. Với chủ repo, có thể dùng token fine-grained có quyền Contents read/write. Với collaborator của repo cá nhân public, có thể dùng PAT classic với scope public_repo. |
| **Google Drive** · khi bắt đầu chạy | Chọn tài khoản khi Colab hỏi. Mặc định archive thuộc thư mục **My Drive/arxiv-m1-benchmark-20k** của người chạy. Mỗi biên nhận trên GitHub có đường dẫn, file ID và checksum. |
| **BENCH_DRIVE_FOLDER** · tùy chọn | Để nhóm cùng mở được raw, chia sẻ một thư mục Drive quyền Editor rồi đặt cùng **ID thư mục** vào Secrets của mỗi người. Nếu dùng Drive riêng, chủ thư mục có thể chia sẻ cho nhóm sau. |
| **S2_API_KEY** · tùy chọn | Nếu có key Semantic Scholar riêng, thêm vào Secrets và bật Notebook access. Mỗi lượt ghi chế độ có/không có key để so sánh kết quả. |

**Kết quả nhóm.** Mở bảng tiến độ ngay cả khi không còn Colab nào chạy. Lần chạy đầu tiên sẽ tự khởi tạo nhánh kết quả nếu cần. [Bảng tiến độ](https://github.com/QuangPhat101/arXiv-Report/blob/benchmark-20k-results/milestone1/data/benchmark_20k/progress.md) · [Hướng dẫn token GitHub](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens)

## Dữ liệu và đơn vị đo

Bốn CSV trong repo báo cáo liệt kê từng **version**. Notebook lấy cột **arxiv_id**, bỏ các dòng lặp của cùng một bài, kiểm tra mỗi tập có 5.000 ID và cả bốn tập có đúng 20.000 ID khác nhau. Mỗi lô chứa 25 ID theo danh sách này. Lô qua ranh giới tháng dùng trực tiếp danh sách ID mà pipeline nhận.

| CSV | ID đầu | ID cuối | Số bài |
|---|---|---|---:|
| arxiv_2503_summary.csv | 2503.01914 | 2503.06913 | 5.000 |
| arxiv_2503_2504_summary.csv | 2503.21914 | 2504.02522 | 5.000 |
| arxiv_2505_summary.csv | 2505.10666 | 2505.15665 | 5.000 |
| arxiv_2506_summary.csv | 2506.05788 | 2506.10787 | 5.000 |

**Thông số chiến dịch.** Cả nhóm dùng cùng commit scraper, 5 download workers, 4 S2 workers và profiler 1 giây. Mỗi lượt dùng thư mục trống. Hai JSON A/B cũ của mẫu 100 bài chỉ được đọc để đối chiếu. Các bảng của chiến dịch 20k bắt đầu từ số 0.

**Nhiều Colab.** Tối đa bốn runtime nhận lô đồng thời. Các truy vấn metadata đến arXiv API dùng chung một lượt truy cập, có khoảng nghỉ giữa hai yêu cầu. Tải source từ export.arxiv.org và truy vấn S2 chạy theo bộ giới hạn của scraper ở mỗi VM. File network.json.gz ghi thời gian chờ phối hợp, số HTTP response theo mã và các lần ngủ của crawler theo hàm. [Điều kiện arXiv API](https://info.arxiv.org/help/api/tou.html) · [Hướng dẫn harvesting](https://info.arxiv.org/help/bulk_data.html)

In [ ]:
#@title 1. Danh sách ID, Git và quyền giữ lô
import csv
import gzip
import hashlib
import io
import json
import os
import platform
import random
import re
import shutil
import signal
import subprocess
import sys
import tarfile
import tempfile
import threading
import time
import uuid
from collections import Counter
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

CODE_REPO = "QuangPhat101/arXiv-Project"
REPORT_REPO = "QuangPhat101/arXiv-Report"
CODE_SHA = "c87f35aca9495f7e3aa4261340d5f10778b692ea"
REPORT_SHA = "50945d301d1939862f4cad5a33a16d4731374e57"
RESULT_BRANCH = "benchmark-20k-results"
NOTEBOOK_BRANCH = "benchmark-20k-notebook"
BASE = Path("milestone1/data/benchmark_20k")
RANGE_FILES = ("arxiv_2503_summary.csv", "arxiv_2503_2504_summary.csv",
               "arxiv_2505_summary.csv", "arxiv_2506_summary.csv")
BLOCK_SIZE, LEASE_S, HEARTBEAT_S = 25, 360, 45
MAX_PASSES = 3                 # Includes the first evaluation of an ID
MAX_ACTIVE_VMS = 4
SCHEMA = 1
PROFILE_KEYS = {"testbed", "run_summary", "memory", "storage", "phase_breakdown_s",
                "paper_records", "failed_papers"}


def pack(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2,
                      allow_nan=False).encode("utf-8") + b"\n"


def load_json(path):
    return json.loads(Path(path).read_bytes())


def utc(ts):
    return datetime.fromtimestamp(ts, timezone.utc).isoformat(timespec="seconds")


def file_digest(path, algorithm="sha256"):
    with open(path, "rb") as stream:
        return hashlib.file_digest(stream, algorithm).hexdigest()


def public_checkout(repo, sha, dest):
    """Only the immutable commit recorded in the notebook is executed."""
    subprocess.run(["git", "init", "-q", str(dest)], check=True)
    url = f"https://github.com/{repo}.git"
    for i in range(4):
        result = subprocess.run(["git", "-C", str(dest), "fetch", "-q", "--depth=1",
                                 url, sha], timeout=180, stderr=subprocess.PIPE, text=True)
        if result.returncode == 0:
            break
        if i == 3:
            raise RuntimeError(f"Git fetch {repo} @ {sha[:10]}: {result.stderr[-350:]}")
        time.sleep(2 ** i)
    subprocess.run(["git", "-C", str(dest), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    actual = subprocess.check_output(["git", "-C", str(dest), "rev-parse", "HEAD"], text=True).strip()
    if actual != sha:
        raise ValueError(f"Commit không khớp ở {repo}: {actual}")


def read_manifest(report):
    groups, sequence, spans = {}, [], {}
    for filename in RANGE_FILES:
        with (report / "milestone1/data" / filename).open(encoding="utf-8-sig", newline="") as src:
            ids = sorted({row["arxiv_id"].strip() for row in csv.DictReader(src)})
        if len(ids) != 5000 or not all(re.fullmatch(r"\d{4}\.\d{5}", x) for x in ids):
            raise ValueError(f"{filename}: cần đúng 5.000 arXiv ID cơ sở")
        sequence.extend(ids)
        spans[filename] = [ids[0], ids[-1], len(ids)]
        for idx in range(0, len(ids), BLOCK_SIZE):
            group = ids[idx:idx + BLOCK_SIZE]
            groups[group[0]] = group
    if len(sequence) != 20000 or len(set(sequence)) != 20000:
        raise ValueError("Các CSV phải chứa đúng 20.000 ID không trùng.")
    manifest = "\n".join(sequence).encode() + b"\n"
    contract = dict(schema=SCHEMA, scraper_repo=CODE_REPO, scraper_sha=CODE_SHA,
                    report_repo=REPORT_REPO, manifest_ref=REPORT_SHA, csv_ranges=spans,
                    id_count=len(sequence), id_list_sha256=hashlib.sha256(manifest).hexdigest(),
                    block_size=BLOCK_SIZE, max_active_vms=MAX_ACTIVE_VMS, retry_passes=MAX_PASSES, download_workers=5,
                    s2_workers=4, profile_interval_s=1, api_slot_s=120, api_gap_s=3.5,
                    runner_sha256=hashlib.sha256(CHILD_SOURCE.encode()).hexdigest())
    return groups, contract, manifest


class GitJournal:
    """Each update has exactly one parent. A non-fast-forward push is recomputed."""
    def __init__(self, directory, remote, branch, base_sha, token=None):
        self.path, self.remote = Path(directory), str(remote)
        self.branch, self.base = branch, base_sha
        self.ref = f"refs/heads/{branch}"
        self.env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_NOSYSTEM="1")
        if token:
            helper = self.path.parent / ("askpass-" + uuid.uuid4().hex)
            helper.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token;; '
                              '*) printf "%s\\n" "$BENCH_PUSH_SECRET";; esac\n')
            helper.chmod(0o700)
            self.env.update(GIT_ASKPASS=str(helper), BENCH_PUSH_SECRET=token)
        self.path.mkdir(parents=True, exist_ok=True)
        self.git("init", "-q")
        self.git("config", "user.name", "Colab benchmark")
        self.git("config", "user.email", "benchmark@users.noreply.github.com")
        self.git("config", "credential.helper", "")
        self.git("config", "gc.auto", "0")
        self.ready = False

    def git(self, *args, check=True):
        p = subprocess.run(["git", "-C", str(self.path), *map(str, args)], env=self.env,
                           stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=100)
        if check and p.returncode:
            raise RuntimeError("Git: " + p.stderr.strip()[-800:])
        return p

    def refresh(self):
        ref = self.ref
        if not self.ready:
            probe = self.git("ls-remote", "--heads", self.remote, ref)
            if not probe.stdout.strip():
                ref = self.base
        self.git("fetch", "-q", "--depth=1", self.remote, ref)
        self.git("reset", "--hard", "-q", "FETCH_HEAD")
        self.git("clean", "-qfd")
        self.ready = ref == self.ref

    def update(self, message, edit):
        for attempt in range(12):
            try:
                self.refresh()
                result, writes = edit(self.path)
                if not writes:
                    return result
                for dest, blob in writes.items():
                    dest = self.path / BASE / dest
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    dest.write_bytes(blob)
                self.git("add", "-f", "--", BASE)
                if self.git("diff", "--cached", "--quiet", check=False).returncode == 0:
                    return result
                self.git("commit", "-q", "-m", message)
                p = self.git("push", "--porcelain", self.remote, f"HEAD:{self.ref}", check=False)
                if p.returncode == 0:
                    self.ready = True
                    return result
                raise RuntimeError((p.stderr + p.stdout)[-850:])
            except (subprocess.TimeoutExpired, RuntimeError) as exc:
                label = str(exc).lower()
                if any(s in label for s in ("403", "401", "denied", "authentication",
                                             "protected branch", "repository rule", "gh013")):
                    raise RuntimeError(f"Kiểm tra quyền đọc/ghi {self.branch}: {exc}") from None
                if attempt == 11:
                    raise
                time.sleep(random.uniform(0.4, min(14, 1.7 ** (attempt + 1))))
        raise AssertionError("unreachable")


def bootstrap(journal, groups, contract, ids):
    def edit(repo):
        path = repo / BASE / "campaign.json"
        if path.exists():
            if load_json(path) != contract:
                raise ValueError("Cấu hình campaign khác bản trên repo. Mở đúng notebook của nhóm.")
            return None, {}
        initial = {"campaign.json": pack(contract), "target_ids.txt": ids}
        initial.update(progress_files(latest_progress(repo, groups, initial)))
        return None, initial
    journal.update("Initialize 20k benchmark manifest", edit)


def fresh_state():
    return {"lease": None, "passes": {}, "papers": {}, "done_at": None}


def get_state(root, block):
    path = root / BASE / "blocks" / block / "state.json"
    return load_json(path) if path.exists() else fresh_state()


def save_state(block, state):
    return {f"blocks/{block}/state.json": pack(state)}


def terminal(item, passes):
    return item["outcome"] in ("SUCCESS", "PDF_ONLY") or passes >= MAX_PASSES


class LeaseLost(ValueError):
    pass


def check_owner(state, token, now):
    lease = state["lease"]
    if not lease or lease["token"] != token or lease["expires"] <= now:
        raise LeaseLost("Lô đã hết hạn hoặc đã được cấp lại.")


def claim(journal, groups, session, now_fn=time.time):
    token = uuid.uuid4().hex
    def edit(repo):
        now = now_fn()
        candidates, earliest, final, owned, active = [], None, 0, None, 0
        for block, ids in groups.items():
            s = get_state(repo, block)
            if s["done_at"] is not None:
                final += len(ids)
                continue
            lease = s["lease"]
            if lease and lease["token"] == token and lease["expires"] > now:
                owned = (block, s)
            if lease and lease["expires"] > now:
                active += 1
                earliest = min(earliest or lease["expires"], lease["expires"])
                continue
            # Recover abandoned work first; then fresh blocks before extra passes.
            rank = 0 if lease else 2 if s["passes"] else 1
            candidates.append((rank, block, s))
        if owned:
            block, s = owned
            return ("work", block, token, s["lease"]["ids"]), {}
        if not candidates or active >= MAX_ACTIVE_VMS:
            return (("complete", final) if earliest is None else
                    ("wait", min(30, max(2, earliest - now + 1)))), {}
        _, block, s = min(candidates, key=lambda e: (e[0], e[1]))
        todo = [i for i in groups[block] if i not in s["papers"] or
                not terminal(s["papers"][i], s["passes"][i])]
        s["lease"] = dict(token=token, session=session, started=now, expires=now+LEASE_S, ids=todo)
        return ("work", block, token, todo), save_state(block, s)
    return journal.update(f"Claim work · {session[:8]}", edit)


def touch(journal, block, token, files, release=False, now_fn=time.time):
    def edit(repo):
        s = get_state(repo, block)
        check_owner(s, token, now_fn())
        if release:
            s["lease"] = None
        else:
            s["lease"]["expires"] = now_fn() + LEASE_S
        return None, {**files, **save_state(block, s)}
    return journal.update(f"{'Release' if release else 'Checkpoint'} {block} · {token[:8]}", edit)


def commit_pass(journal, groups, block, token, rows, files, archive, now_fn=time.time):
    marker = f"attempts/{block}/{token}/commit.json"
    def edit(repo):
        if (repo / BASE / marker).is_file():
            return None, {}                   # Ambiguous acknowledgement after a successful push
        s = get_state(repo, block)
        check_owner(s, token, now_fn())
        expected = s["lease"]["ids"]
        if set(rows) != set(expected) or len(rows) != len(expected):
            raise ValueError("Các ID đã đo khác với danh sách được cấp.")
        if not archive or not archive.get("drive_id") or not archive.get("sha256"):
            raise ValueError("Thiếu biên nhận archive.")
        for aid, item in rows.items():
            s["passes"][aid] = s["passes"].get(aid, 0) + 1
            s["papers"][aid] = {k: item[k] for k in ("outcome", "raw_bytes", "extracted_bytes", "disk_bytes")}
            s["papers"][aid]["attempt"] = token
        s["lease"] = None
        if all(i in s["papers"] and terminal(s["papers"][i], s["passes"][i]) for i in groups[block]):
            s["done_at"] = now_fn()
        receipt = dict(committed_at=now_fn(), ids=expected, archive=archive)
        writes = {**files, marker: pack(receipt), **save_state(block, s)}
        writes.update(progress_files(latest_progress(repo, groups, writes)))
        return None, writes
    journal.update(f"Commit {len(rows)} measured IDs · {block} · {token[:8]}", edit)


def api_slot(journal, block, token, now_fn=time.time):
    """One short-lived discovery slot for the whole team; downloads proceed separately."""
    def edit(repo):
        now = now_fn()
        check_owner(get_state(repo, block), token, now)
        path = repo / BASE / "api-slot.json"
        item = load_json(path) if path.exists() else {}
        if item.get("owner") != token and item.get("until", 0) + 4 > now:
            return (False, item["until"] + 4), {}
        if item.get("owner") == token and item["until"] > now + 40:
            return (True, item["until"]), {}
        until = now + 120
        return (True, until), {"api-slot.json": pack(dict(owner=token, until=until))}
    return journal.update(f"Reserve arXiv discovery · {token[:8]}", edit)


def free_slot(journal, token, now_fn=time.time):
    def edit(repo):
        p = repo / BASE / "api-slot.json"
        if p.exists() and load_json(p).get("owner") == token:
            return None, {"api-slot.json": pack(dict(owner=None, until=now_fn()))}
        return None, {}
    journal.update(f"Close arXiv discovery · {token[:8]}", edit)

In [ ]:
#@title 2. Tiến trình đo scraper
CHILD_SOURCE = r'''"""Run the pinned scraper in its own process so its RAM/CPU exclude the notebook."""
import functools
import json
import logging
import os
import re
import sys
import threading
import time
import uuid
import ctypes
import signal
from collections import Counter
from pathlib import Path
from urllib.parse import urlsplit

import arxiv
import requests
from src import crawler, discovery, references, utils

# Colab runs Linux. Hard kernel restarts terminate a child even if the VM survives.
parent_pid = int(sys.argv[2])
libc = ctypes.CDLL(None, use_errno=True)
if libc.prctl(1, signal.SIGTERM, 0, 0, 0) != 0:   # PR_SET_PDEATHSIG
    raise OSError(ctypes.get_errno(), "could not arm parent-death signal")
if os.getppid() != parent_pid:
    sys.exit(75)

work = Path(sys.argv[1])
ids = json.loads((work / "ids.json").read_text())
api_dir = work / "api"
api_dir.mkdir(exist_ok=True)
output = work / "output"
meter_lock = threading.Lock()
timers = Counter()
counts = Counter()
last_api = 0.0
api_lock = threading.Lock()
active_profiler = None


def write_json(path, value):
    temp = path.with_name(path.name + ".tmp")
    temp.write_text(json.dumps(value, ensure_ascii=False))
    temp.replace(path)


def count(name, seconds=None):
    with meter_lock:
        counts[name] += 1
        if seconds is not None:
            timers[name] += seconds


class Clock:
    """Observe actual sleeps in each crawler module without changing the stdlib clock."""
    def __init__(self, module):
        self.module = module

    def __getattr__(self, name):
        return getattr(time, name)

    def sleep(self, seconds):
        frame = sys._getframe(1)
        name = f"{self.module}.{frame.f_code.co_name}"
        start = time.monotonic()
        try:
            time.sleep(seconds)
        finally:
            count("sleep." + name, time.monotonic() - start)


for module in (utils, references, discovery, arxiv):
    if hasattr(module, "time") and module.time is time:
        module.time = Clock(module.__name__)


original_send = requests.adapters.HTTPAdapter.send


def network_call(adapter, request, kwargs, domain):
    start = time.monotonic()
    try:
        response = original_send(adapter, request, **kwargs)
        count(f"http.{domain}.{response.status_code}")
        return response
    except Exception as exc:
        count(f"http.{domain}.error.{type(exc).__name__}")
        raise
    finally:
        count(f"request.{domain}", time.monotonic() - start)


def measured_send(adapter, request, **kwargs):
    global last_api
    url = urlsplit(request.url)
    host = url.hostname or ""
    is_arxiv = host == "arxiv.org" or host.endswith(".arxiv.org")
    is_api = is_arxiv and url.path.rstrip("/") == "/api/query"
    if not is_api:
        domain = "source" if is_arxiv else "s2" if host.endswith("semanticscholar.org") else "other"
        return network_call(adapter, request, kwargs, domain)
    # Physical HTTP requests, including retries, all obtain a team-wide permit.
    with api_lock:
        start = time.monotonic()
        key = uuid.uuid4().hex
        want, grant, done = [api_dir / (key + ext) for ext in (".want", ".grant", ".done")]
        want.touch()
        while not grant.exists():
            time.sleep(0.2)
        until = json.loads(grant.read_text())
        delay = max(0.0, 3.5 - (time.monotonic() - last_api))
        if delay:
            time.sleep(delay)
        if until - time.time() < 35:
            done.touch()
            raise TimeoutError("Discovery permit expired before dispatch")
        count("coord.discovery_wait", time.monotonic() - start)
        kwargs["timeout"] = (8, 20)
        # A paused parent or stalled response cannot hold a live connection beyond its lease.
        deadline = threading.Timer(max(1.0, until - time.time() - 5), os._exit, args=(75,))
        deadline.daemon = True
        deadline.start()
        try:
            last_api = time.monotonic()
            response = network_call(adapter, request, kwargs, "arxiv_api")
            _ = response.content             # finish reading while this VM owns the connection
            return response
        finally:
            deadline.cancel()
            done.touch()


requests.adapters.HTTPAdapter.send = measured_send
original_discover = crawler.discover_paper


@functools.wraps(original_discover)
def discover_or_raise(*args, **kwargs):
    # The producer has an exception fallback but forwards None to a download worker.
    # Route an empty lookup to that existing fallback; the parent flags its empty title.
    meta = original_discover(*args, **kwargs)
    if not meta or not meta.get("versions"):
        raise LookupError("arXiv discovery returned no versions")
    return meta


crawler.discover_paper = discover_or_raise


class ObservedProfiler(crawler.CrawlerProfiler):
    def start(self):
        global active_profiler
        super().start()
        active_profiler = self


crawler.CrawlerProfiler = ObservedProfiler
stop_snapshot = threading.Event()


def snapshot():
    with meter_lock:
        measures = {"counts": dict(counts), "seconds": dict(timers)}
    write_json(work / "network.json", measures)
    if active_profiler is not None:
        write_json(work / "partial.json", active_profiler.get_summary())


def monitor():
    while not stop_snapshot.wait(8):
        snapshot()


logging.basicConfig(level=logging.INFO, format="%(asctime)sZ %(levelname)s %(name)s %(message)s")
logging.Formatter.converter = time.gmtime
class RetryEvents(logging.Handler):
    def emit(self, record):
        msg = record.getMessage().lower()
        if (record.name in {"src.references", "src.discovery", "src.downloader", "src.utils"}
                and re.search(r"\bretrying\b|\bbefore retry\b", msg)):
            count("retry_notice." + record.name)
        elif record.name == "arxiv" and msg.startswith("requesting page") and re.search(r"try: [1-9]", msg):
            count("retry_notice.arxiv_client")


logging.getLogger().addHandler(RetryEvents())
sampler = threading.Thread(target=monitor, daemon=True)
sampler.start()
try:
    summary = crawler.run_pipeline(
        arxiv_ids=ids, output_dir=output, resume=False, batch_size=100,
        s2_api_key=os.environ.pop("BENCH_S2", None), profile=True, profile_output=work/"stats.json",
        n_workers=5, download_workers=5, s2_workers=4, baseline=False)
    write_json(work / "pipeline.json", summary)
finally:
    stop_snapshot.set()
    sampler.join()
    snapshot()
'''

In [ ]:
#@title 3. Kiểm tra dữ liệu, lưu Drive và đồng bộ theo lô
def secret(name):
    from google.colab import userdata
    try:
        return userdata.get(name)
    except userdata.SecretNotFoundError:
        return None
    except userdata.NotebookAccessError:
        raise ValueError(f"Bật Notebook access cho {name} trong Colab Secrets.") from None


def archive_bytes(root):
    """Logical bytes including in-flight archives. Files may disappear during a sample."""
    size = 0
    for directory, _, files in os.walk(root):
        for name in files:
            try:
                size += (Path(directory) / name).stat().st_size
            except FileNotFoundError:
                pass
    return size


class DriveArchive:
    def __init__(self, folder=None):
        from google.colab import auth
        from googleapiclient.discovery import build
        auth.authenticate_user()
        self.drive = build("drive", "v3", cache_discovery=False).files()
        if folder:
            item = self.drive.get(fileId=folder, supportsAllDrives=True,
                                  fields="id,mimeType,trashed,capabilities(canAddChildren)").execute(num_retries=4)
            if (item["mimeType"] != "application/vnd.google-apps.folder" or item.get("trashed")
                    or not item.get("capabilities", {}).get("canAddChildren")):
                raise ValueError("BENCH_DRIVE_FOLDER cần là thư mục Google Drive có quyền thêm file.")
            self.folder = item["id"]
        else:
            name = "arxiv-m1-benchmark-20k"
            mime = "application/vnd.google-apps.folder"
            found = self.drive.list(q=f"'root' in parents and name='{name}' and mimeType='{mime}' and trashed=false",
                                    fields="files(id)", pageSize=10).execute(num_retries=4)["files"]
            self.folder = found[0]["id"] if found else self.drive.create(
                body=dict(name=name, mimeType=mime), fields="id").execute(num_retries=4)["id"]

    def put(self, bundle):
        from googleapiclient.http import MediaFileUpload
        file_id = self.drive.generateIds(count=1, space="drive").execute(num_retries=4)["ids"][0]
        md5 = file_digest(bundle, "md5")
        fields = "id,size,md5Checksum,webViewLink"
        for n in range(4):
            try:
                media = MediaFileUpload(str(bundle), mimetype="application/gzip", resumable=True,
                                        chunksize=8 * 1024 ** 2)
                req = self.drive.create(body=dict(id=file_id, name=bundle.name, parents=[self.folder]),
                                        media_body=media, fields=fields, supportsAllDrives=True)
                result = None
                while result is None:
                    _, result = req.next_chunk(num_retries=3)
                break
            except Exception:
                # Remote may have committed the upload before the acknowledgement was lost.
                try:
                    result = self.drive.get(fileId=file_id, fields=fields,
                                            supportsAllDrives=True).execute(num_retries=2)
                    break
                except Exception:
                    if n == 3:
                        raise
                    time.sleep(2 ** n)
        if result.get("md5Checksum") != md5 or int(result["size"]) != bundle.stat().st_size:
            raise IOError("Checksum hoặc kích thước của file trên Drive không khớp.")
        return dict(drive_id=result["id"], drive_folder=self.folder, name=bundle.name,
                    url=result.get("webViewLink", f"https://drive.google.com/open?id={file_id}"),
                    md5=md5, sha256=file_digest(bundle), size_bytes=bundle.stat().st_size)


def measured_rows(stats, expected, output):
    if not PROFILE_KEYS.issubset(stats):
        raise ValueError("Profiler thiếu trường bắt buộc.")
    records = stats["paper_records"]
    ids = [r["arxiv_id"] for r in records]
    run = stats["run_summary"]
    if (len(ids) != len(expected) or set(ids) != set(expected) or
            run["evaluated_papers"] != len(ids) or not stats["memory"]["sample_count"]):
        raise ValueError("Profiler thiếu mẫu/ID, có ID trùng hoặc sai dải.")
    results = {}
    for rec in records:
        aid, outcome = rec["arxiv_id"], rec["status"]
        if outcome not in {"SUCCESS", "PDF_ONLY", "FAILED"}:
            raise ValueError(f"Trạng thái lạ: {outcome}")
        paper = output / aid
        reason = rec.get("error")
        if outcome != "FAILED":
            try:
                meta = load_json(paper / "metadata.json")
                versions = meta["versions"]
                if (meta.get("arxiv_id") != aid or not meta.get("title", "").strip()
                        or not isinstance(versions, list) or not versions
                        or not isinstance(load_json(paper / "references.json"), list)):
                    raise ValueError("metadata/versions/references không đầy đủ")
                for vid in versions:
                    if not re.fullmatch(re.escape(aid) + r"v[1-9]\d*", vid):
                        raise ValueError("version ID không hợp lệ")
                    d = paper / vid[len(aid):]
                    stamp = (d / ".complete").read_text().strip().split("|")[0]
                    if stamp == "PDF_ONLY":
                        continue
                    if stamp != "SUCCESS" or not any(
                        f.is_file() and f.suffix.lower() in (".tex", ".bib") for f in d.rglob("*")
                    ):
                        raise ValueError("thiếu nguồn hoặc dấu hoàn tất version")
            except (OSError, ValueError, KeyError, TypeError, AttributeError) as exc:
                outcome, reason = "INCOMPLETE", str(exc)
        results[aid] = dict(outcome=outcome, reason=reason, disk_bytes=archive_bytes(paper),
                            raw_bytes=rec["raw_bytes"], extracted_bytes=rec["extracted_bytes"])
    return results


def make_bundle(work, block, token, ids):
    archive = work / f"{block}-{token}.tar.gz"
    with tarfile.open(archive, "w:gz", compresslevel=3, dereference=False) as tar:
        for aid in ids:
            directory = work / "output" / aid
            if directory.is_dir():
                tar.add(directory, arcname=aid)
        data = pack({"block": block, "attempt": token, "ids": ids, "scraper_sha": CODE_SHA})
        desc = tarfile.TarInfo("batch.json")
        desc.size = len(data)
        tar.addfile(desc, io.BytesIO(data))
    return archive


def atomic_write(path, blob):
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_bytes(blob)
    tmp.replace(path)


class DiscoveryPermits:
    def __init__(self, journal, block, token, folder):
        self.journal, self.block, self.token, self.folder = journal, block, token, folder
        self.active, self.next_check, self.finished = None, 0.0, set()

    def service(self):
        if self.active:
            if not self.active.with_suffix(".done").exists():
                return
            free_slot(self.journal, self.token)
            self.finished.add(self.active)
            self.active = None
        if time.monotonic() < self.next_check:
            return
        for want in sorted(self.folder.glob("*.want")):
            if want in self.finished:
                continue
            ok, until = api_slot(self.journal, self.block, self.token)
            if ok:
                atomic_write(want.with_suffix(".grant"), pack(until))
                self.active = want
            else:
                self.next_check = time.monotonic() + min(12, max(2, until - time.time()))
            break


def collect_files(work, prefix, info, keys):
    data = {prefix + "observed.json": pack(info)}
    for name in keys:
        p = work / name
        if p.is_file():
            payload = p.read_bytes()
            for credential in REDACTIONS:
                if credential:
                    payload = payload.replace(credential.encode(), b"[redacted]")
            data[prefix + name + ".gz"] = gzip.compress(payload, compresslevel=3, mtime=0)
    return data


def stop_child(proc):
    if proc is not None and proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGTERM)
            proc.wait(timeout=4)
        except subprocess.TimeoutExpired:
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait()
        except ProcessLookupError:
            pass


def deps_info():
    values = {}
    for name in ("arxiv", "requests", "psutil", "tqdm", "google-api-python-client"):
        try:
            values[name] = metadata.version(name)
        except metadata.PackageNotFoundError:
            values[name] = None
    return values


def run_attempt(journal, groups, job, code, archive_store, scratch, session, s2):
    import psutil
    _, block, token, ids = job
    prefix = f"attempts/{block}/{token}/"
    with tempfile.TemporaryDirectory(prefix="attempt-", dir=scratch) as tmpdir:
        work = Path(tmpdir)
        (work / "ids.json").write_bytes(pack(ids))
        (work / "runner.py").write_text(CHILD_SOURCE)
        (work / "api").mkdir()
        permits = DiscoveryPermits(journal, block, token, work / "api")
        observed = dict(block=block, session=session, attempt=token, ids=ids,
                        scraper_sha=CODE_SHA, packages=deps_info(), s2_key_present=bool(s2),
                        platform=platform.platform(), architecture=platform.machine(),
                        python=platform.python_version(), cpu_count=os.cpu_count(),
                        cpu_affinity=len(os.sched_getaffinity(0)), ram_bytes=psutil.virtual_memory().total,
                        disk_free_start_bytes=shutil.disk_usage(work).free,
                        phase="claimed", started_utc=time.time(), elapsed_observed_s=0.0,
                        disk_peak_sampled_bytes=0)
        live_files = ["partial.json", "network.json", "crawler.log", "disk.jsonl"]
        first_expiry_floor = time.time() + LEASE_S - 30
        touch(journal, block, token, collect_files(work, prefix, observed, []))
        proc, monitor = None, None
        origin = time.monotonic()
        next_sync = origin + HEARTBEAT_S
        # This host can keep its kernel after stopping the cell. Kill the child if the
        # last remotely acknowledged lease could expire while network retries block.
        deadline_stop = threading.Event()
        valid_until = [first_expiry_floor]
        def watch_lease():
            while not deadline_stop.wait(1):
                if time.time() >= valid_until[0]:
                    stop_child(proc)
                    return
        watchdog = threading.Thread(target=watch_lease, daemon=True)
        sample_lock = threading.Lock()
        stop_disk = threading.Event()
        def sample_disk():
            with sample_lock:
                size = archive_bytes(work / "output")
                observed["disk_peak_sampled_bytes"] = max(observed["disk_peak_sampled_bytes"], size)
                with (work / "disk.jsonl").open("a") as f:
                    f.write(json.dumps([round(time.monotonic()-origin, 3), size]) + "\n")
        def disk_loop():
            while not stop_disk.wait(2):
                sample_disk()
        def checkpoint(force=False):
            nonlocal next_sync
            now = time.monotonic()
            if force or now >= next_sync:
                observed["elapsed_observed_s"] = round(now - origin, 3)
                observed["last_seen_utc"] = time.time()
                files = collect_files(work, prefix, observed, live_files)
                expiry_floor = time.time() + LEASE_S - 30
                touch(journal, block, token, files)
                valid_until[0] = expiry_floor
                next_sync = time.monotonic() + HEARTBEAT_S
        observed["started_utc"] = time.time()
        observed["phase"] = "running"
        try:
            env = dict(os.environ, PYTHONPATH=str(code), PYTHONUNBUFFERED="1", TZ="UTC")
            if s2:
                env["BENCH_S2"] = s2
            with (work/"crawler.log").open("wb") as log:
                proc = subprocess.Popen([sys.executable, "-u", str(work/"runner.py"), str(work), str(os.getpid())],
                                        env=env, cwd=code, stdout=log, stderr=subprocess.STDOUT,
                                        start_new_session=True)
                watchdog.start()
                monitor = threading.Thread(target=disk_loop, daemon=True)
                monitor.start()
                while proc.poll() is None:
                    checkpoint()
                    permits.service()
                    if time.monotonic() - origin > max(3600, 180 * len(ids)):
                        raise TimeoutError("Crawler vượt thời gian tối đa của một lô.")
                    time.sleep(0.4)
            permits.service()
            stop_disk.set()
            monitor.join()
            sample_disk()
            observed["child_exit"] = proc.returncode
            observed["child_elapsed_s"] = round(time.monotonic() - origin, 3)
            if proc.returncode != 0:
                raise ValueError(f"Crawler exit {proc.returncode}; xem crawler.log.gz.")
            stats = load_json(work / "stats.json")
            rows = measured_rows(stats, ids, work / "output")
            mem = stats["memory"]
            net = load_json(work / "network.json")
            observed["profile"] = dict(count=len(rows), wall_s=stats["run_summary"]["wall_time_s"],
                                       phases_s=stats["phase_breakdown_s"],
                                       waits_s=net["seconds"], network_counts=net["counts"],
                                       ram_peak_mib=mem["peak_ram_mb"], ram_avg_mib=mem["avg_ram_mb"],
                                       ram_samples=mem["sample_count"])
            observed["phase"] = "uploading"
            observed["disk_final_bytes"] = archive_bytes(work / "output")
            live_files.extend(["stats.json", "pipeline.json"])
            checkpoint(force=True)

            # Keep the lease alive even while a slow Drive upload or compression is in flight.
            from concurrent.futures import ThreadPoolExecutor, wait
            pool = ThreadPoolExecutor(max_workers=1)
            try:
                future = pool.submit(lambda: archive_store.put(make_bundle(work, block, token, ids)))
                while not wait([future], timeout=3).done:
                    checkpoint()
                archive = future.result()
            finally:
                pool.shutdown(wait=True)
            # The native report supersedes its rolling preview.
            observed.update(phase="committed", finished_utc=time.time(),
                            last_seen_utc=time.time(), elapsed_observed_s=round(time.monotonic()-origin, 3))
            files = collect_files(work, prefix, observed, live_files)
            files[prefix+"outcomes.json"] = pack(rows)
            commit_pass(journal, groups, block, token, rows, files, archive)
            print(f"✓ {block} · {len(ids)} ID · {dict(Counter(r['outcome'] for r in rows.values()))}", flush=True)
        except BaseException as exc:
            stop_child(proc)
            observed.update(phase="interrupted", last_seen_utc=time.time(),
                            elapsed_observed_s=round(time.monotonic()-origin, 3),
                            exception=f"{type(exc).__name__}: {str(exc)[:400]}")
            try:
                touch(journal, block, token, collect_files(work, prefix, observed, live_files), release=True)
            except (LeaseLost, RuntimeError, OSError, subprocess.TimeoutExpired):
                pass                         # Last confirmed heartbeat remains recoverable
            raise
        finally:
            deadline_stop.set()
            if watchdog.ident is not None:
                watchdog.join(timeout=5)
            stop_disk.set()
            if monitor is not None:
                monitor.join(timeout=5)
            stop_child(proc)
            if permits.active:
                free_slot(journal, token)

In [ ]:
#@title 4. Tổng hợp và vẽ biểu đồ benchmark
_OBS_CACHE = {}


def latest_progress(repo, groups, changes=None):
    changes = changes or {}
    root = repo / BASE
    def read(name):
        return json.loads(changes[name]) if name in changes else load_json(root / name)
    def paths(pattern, suffix):
        found = {str(p.relative_to(root)) for p in root.glob(pattern)}
        return sorted(found | {p for p in changes if p.startswith("attempts/") and p.endswith(suffix)})

    spans = read("campaign.json")["csv_ranges"]
    range_rows, final, distinct, total_disk = {}, Counter(), 0, 0
    raw, extracted = 0, 0
    for b, ids in groups.items():
        rel = f"blocks/{b}/state.json"
        s = read(rel) if rel in changes or (root / rel).exists() else fresh_state()
        label = s.get("range") or next(k for k, (lo, hi, _) in spans.items()
                                       if lo <= b <= hi)
        row = range_rows.setdefault(label, Counter())
        row["target"] += len(ids)
        for aid, item in s["papers"].items():
            mark = item["outcome"]
            distinct += 1
            row["evaluated"] += 1
            row[mark] += 1
            final[mark] += 1
            raw += item.get("raw_bytes", 0)
            extracted += item.get("extracted_bytes", 0)
            total_disk += item.get("disk_bytes", 0)
            if terminal(item, s["passes"][aid]):
                row["settled"] += 1
    def observation(p):
        cache_key = (str(root), p)
        if p in changes or cache_key not in _OBS_CACHE:
            value = read(p)
            # Only completed observations are immutable. Edits staged for CAS are
            # cached after they exist in the checkout.
            if p not in changes and value["phase"] in ("committed", "interrupted"):
                _OBS_CACHE[cache_key] = value
            return value
        return _OBS_CACHE[cache_key]
    runs = [observation(p) for p in paths("attempts/*/*/observed.json", "/observed.json")]
    done = [r for r in runs if r["phase"] == "committed"]
    profiler_seconds = sum(r["profile"]["wall_s"] for r in done)
    samples = sum(r["profile"]["ram_samples"] for r in done)
    record_count = sum(r["profile"]["count"] for r in done)
    observed_seconds = sum(r["elapsed_observed_s"] for r in runs)
    starts = [r["started_utc"] for r in runs]
    stops = [r.get("last_seen_utc", r["started_utc"]) for r in runs]
    settled = sum(x["settled"] for x in range_rows.values())
    phases, waits, counters = Counter(), Counter(), Counter()
    for r in done:
        phases.update(r["profile"]["phases_s"])
        waits.update(r["profile"]["waits_s"])
        counters.update(r["profile"]["network_counts"])
    uncommitted = [r for r in runs if r["phase"] != "committed"]
    return dict(status="finished" if settled == sum(map(len, groups.values())) else "in_progress",
                target=sum(map(len, groups.values())), evaluated_distinct=distinct, settled=settled,
                by_latest_outcome=dict(final), ranges=range_rows, attempts_observed=len(runs),
                committed_passes=len(done), uncommitted_attempts=len(uncommitted),
                profiler_id_observations_including_retries=record_count,
                accepted_profiler_seconds=round(profiler_seconds, 3),
                accepted_profiler_seconds_per_observation=round(profiler_seconds/record_count, 4) if record_count else None,
                all_attempt_machine_seconds_observed=round(observed_seconds, 3),
                uncommitted_machine_seconds_observed=round(sum(r["elapsed_observed_s"] for r in uncommitted), 3),
                calendar_span_observed_s=round(max(stops)-min(starts), 3) if runs else None,
                start_utc=utc(min(starts)) if starts else None, last_observed_utc=utc(max(stops)) if starts else None,
                peak_rss_mib=max((r["profile"]["ram_peak_mib"] for r in done), default=None),
                sample_weighted_rss_mib=round(sum(r["profile"]["ram_avg_mib"]*r["profile"]["ram_samples"]
                                                  for r in done)/samples, 3) if samples else None,
                per_vm_output_disk_peak_sampled_bytes=max((r["disk_peak_sampled_bytes"] for r in runs), default=None),
                latest_id_output_disk_bytes=total_disk, latest_id_raw_bytes=raw,
                latest_id_extracted_bytes=extracted,
                latest_id_savings_pct=round(100*(1-extracted/raw), 3) if raw else None,
                accepted_phases_s=dict(phases), accepted_instrumented_seconds=dict(waits),
                accepted_network_counts=dict(counters), scraper_sha=CODE_SHA)


def progress_files(summary):
    S = summary
    rows = ["# Benchmark arXiv trên Colab", "",
            f"**{S['evaluated_distinct']:,} / {S['target']:,} ID đã đo** · "
            f"{S['settled']:,} ID đã chốt · {S['committed_passes']} lượt lô đã đồng bộ.", "",
            "| Nguồn ID | Đã đo | Đã chốt | SUCCESS | PDF_ONLY | FAILED | INCOMPLETE |",
            "|---|---:|---:|---:|---:|---:|---:|"]
    for k, v in sorted(S["ranges"].items()):
        rows.append("| " + k + " | " + " | ".join(str(v.get(col, 0)) for col in
                    ("evaluated", "settled", "SUCCESS", "PDF_ONLY", "FAILED", "INCOMPLETE")) + " |")
    rows.extend(["", f"- Tổng thời gian profiler của các lượt hợp lệ (gồm lượt thử lại) · {S['accepted_profiler_seconds']:,.1f} s",
                 f"- Tổng thời gian máy quan sát được, kể cả lượt dở và lưu Drive/Git · {S['all_attempt_machine_seconds_observed']:,.1f} s",
                 "- Khoảng thời gian lịch từ lúc bắt đầu đến mẫu cuối (gồm lúc nhóm nghỉ) · "
                 + (f"{S['calendar_span_observed_s']:,.1f} s" if S["calendar_span_observed_s"] is not None else "chưa có mẫu"),
                 f"- Kết quả theo ID, danh tính phiên và biên nhận Drive nằm trong `blocks/` và `attempts/`.",
                 "- Số liệu cập nhật sau mỗi lần chốt lô. Heartbeat của lượt đang chạy nằm trong thư mục attempts.",
                 "- FAILED/INCOMPLETE được giữ nguyên sau khi dùng hết số lượt thử. PDF_ONLY theo định nghĩa profiler của commit này.",
                 "", "Mở `progress.json` để đọc thời gian các pha, RAM, dung lượng, số lần gọi mạng và thời gian chờ.",
                 ""])
    return {"progress.json": pack(summary), "progress.md": "\n".join(rows).encode()}


def dashboard(journal, groups, report, show_figures=True):
    import pandas as pd
    from IPython.display import display
    journal.refresh()
    s = latest_progress(journal.path, groups)
    print(f"{s['evaluated_distinct']:,}/{s['target']:,} ID đã đo · {s['settled']:,} đã chốt"
          f" · {s['uncommitted_attempts']} lượt chưa chốt")
    display(pd.DataFrame(s["ranges"]).T.fillna(0).astype(int))
    display(pd.Series({k:s[k] for k in ("accepted_profiler_seconds", "all_attempt_machine_seconds_observed",
             "calendar_span_observed_s", "peak_rss_mib", "sample_weighted_rss_mib",
             "per_vm_output_disk_peak_sampled_bytes", "latest_id_output_disk_bytes", "latest_id_savings_pct")},
             name="Giá trị").to_frame())
    historic = []
    for name in ("baseline", "parallel"):
        d = load_json(report / "milestone1/data" / f"{name}_colab_stats.json")
        a = d["run_summary"]
        historic.append(dict(profile=name, n=a["evaluated_papers"], seconds=a["wall_time_s"],
                             seconds_per_id=a["avg_time_per_paper_s"], failed=a["failed_count"]))
    display(pd.DataFrame(historic).set_index("profile").rename_axis("A/B cũ · 100 ID cùng dải"))
    if not show_figures:
        return s
    root = journal.path / BASE
    accepted = sorted(root.glob("attempts/*/*/commit.json"),
                      key=lambda p: load_json(p)["committed_at"])
    if accepted:
        import matplotlib.pyplot as plt
        with gzip.open(accepted[-1].with_name("stats.json.gz"), "rt", encoding="utf-8") as f:
            last = json.load(f)
        mem = last["memory"]
        fig, ax = plt.subplots(1, 3, figsize=(14, 3.3), layout="constrained")
        ax[0].plot(mem["timeline_s"], mem["ram_timeline_mb"])
        ax[0].set(xlabel="Giây", ylabel="RSS (MiB)")
        ax[1].plot(mem["timeline_s"], mem["cpu_timeline_pct"])
        ax[1].set(xlabel="Giây", ylabel="CPU (%)")
        phase = s["accepted_phases_s"]
        ax[2].barh(list(phase), [phase[k] for k in phase])
        ax[2].set_xlabel("Giây cộng dồn các pha")
        fig.suptitle("RSS, CPU của lượt cuối · Tổng thời gian pha của các lượt đã chốt")
        plt.show()
    return s

In [ ]:
#@title 5. Tải đúng commit và kết nối hàng đợi của nhóm
if not Path("/content").is_dir():
    raise RuntimeError("Mở notebook bằng Google Colab và chọn CPU runtime để chạy benchmark thật.")
SCRATCH = Path(tempfile.mkdtemp(prefix="arxiv20k-", dir="/content"))
CODE, REPORT = SCRATCH / "scraper", SCRATCH / "report"
public_checkout(CODE_REPO, CODE_SHA, CODE)
public_checkout(REPORT_REPO, REPORT_SHA, REPORT)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                "-r", str(CODE / "src/requirements.txt"), "arxiv==4.0.2",
                "google-api-python-client"], check=True)
GROUPS, CONTRACT, MANIFEST = read_manifest(REPORT)
SESSION = uuid.uuid4().hex
GITHUB_KEY = secret("GITHUB_TOKEN")
if not GITHUB_KEY:
    raise ValueError(f"Thêm GITHUB_TOKEN vào Colab Secrets và bật Notebook access. Cần quyền push {REPORT_REPO}.")
S2_KEY = secret("S2_API_KEY")
REDACTIONS = tuple(s for s in (GITHUB_KEY, S2_KEY) if s)
JOURNAL = GitJournal(SCRATCH/"journal", f"https://github.com/{REPORT_REPO}.git",
                     RESULT_BRANCH, REPORT_SHA, GITHUB_KEY)
bootstrap(JOURNAL, GROUPS, CONTRACT, MANIFEST)
print(f"Scraper {CODE_SHA[:12]} · {len(GROUPS)} lô · {CONTRACT['id_count']:,} ID · "
      f"session {SESSION[:8]} · S2 {'có key' if S2_KEY else 'public'}")
print(f"Kết quả: https://github.com/{REPORT_REPO}/tree/{RESULT_BRANCH}/{BASE}")

In [ ]:
#@title 6. Tự nhận và xử lý các lô còn lại
initial = dashboard(JOURNAL, GROUPS, REPORT, show_figures=False)
if initial["status"] != "finished":
    DRIVE_STORE = DriveArchive(secret("BENCH_DRIVE_FOLDER"))
    failure_streak = 0
    while True:
        job = claim(JOURNAL, GROUPS, SESSION)
        if job[0] == "complete":
            break
        if job[0] == "wait":
            print("Các lô khả dụng đang có người chạy. Phiên này sẽ tự kiểm tra lại.", flush=True)
            time.sleep(job[1] + random.uniform(0, 3))
            continue
        print(f"▶ {job[1]} · {len(job[3])} ID · lưu checkpoint khoảng mỗi {HEARTBEAT_S} giây.", flush=True)
        try:
            run_attempt(JOURNAL, GROUPS, job, CODE, DRIVE_STORE, SCRATCH, SESSION, S2_KEY)
            failure_streak = 0
        except LeaseLost:
            print("Quyền giữ lô đã chuyển. Đọc lại hàng đợi.", flush=True)
        except (OSError, RuntimeError, ValueError, subprocess.TimeoutExpired) as exc:
            failure_streak += 1
            if failure_streak >= 3:
                raise RuntimeError("Phiên gặp lỗi hạ tầng 3 lần liên tiếp. Xem observed.json và crawler.log.gz; "
                                   "sau khi khắc phục, bấm Run all để tiếp tục.") from exc
            print(f"Lỗi lượt chạy ({type(exc).__name__}). Thử lại sau {15*failure_streak}s.", flush=True)
            time.sleep(15*failure_streak)

## Đọc kết quả và phục hồi

Trong nhánh **benchmark-20k-results**, mở **milestone1/data/benchmark_20k/**. Có thể đọc các file nén bằng gzip hoặc pandas/Python.

| Tệp | Dùng khi |
|---|---|
| **campaign.json**, **target_ids.txt** | Xác định chính xác 20.000 ID, commit scraper, danh sách CSV và cấu hình. |
| **progress.md**, **progress.json** | Xem số bài đã đo và đã chốt theo dải, tổng thời gian thực thi, RSS, dung lượng và các pha. Bảng cập nhật trong cùng commit chốt lô. |
| **blocks/ID-đầu/state.json** | Tìm token của lượt gần nhất cho từng ID, số lượt đã đo và trạng thái. |
| **attempts/ID-đầu/token/stats.json.gz**, **pipeline.json.gz** | JSON gốc từ scraper có testbed, thông số worker, wall time, RSS/CPU timeline, thời gian theo pha, lỗi, byte raw/extracted, mức tiết kiệm và paper_records. |
| **crawler.log.gz**, **network.json.gz**, **disk.jsonl.gz**, **partial.json.gz**, **observed.json** | Log, retry/backoff, HTTP, số thông báo retry và số byte trong thư mục output gồm cả tệp tải tạm (lấy mẫu mỗi 2 giây), cấu hình và heartbeat của phiên. Log và telemetry đang chạy được đồng bộ khoảng mỗi 45 giây khi GitHub khả dụng. |
| **outcomes.json**, **commit.json** | Kết quả kiểm tra từng ID, danh sách ID đã chốt, đường dẫn archive Drive, MD5 và SHA-256. Bên trong archive có metadata, references và thư mục version mà scraper thực tế tạo. |

**Nếu phiên tắt đột ngột.** Sau 6 phút kể từ lần gia hạn cuối, một runtime khác có thể nhận lô. Các lô đã chốt vẫn giữ nguyên. Lô đang làm có thể phải đo lại tối đa 25 ID. Nếu VM cũ quay lại trễ, chỉ VM đang giữ lô mới có thể chốt kết quả. File Drive đã lên nhưng chưa có commit có thể còn trong thư mục của người chạy. Bảng tiến độ chỉ dùng archive có biên nhận đã chốt.

**Đã đo và đã chốt.** Một lượt chỉ được ghi nhận khi có đúng một bản ghi cho mỗi ID được giao, archive đã có checksum từ Drive và việc ghi Git thành công. SUCCESS và PDF_ONLY được chốt ngay nếu output qua kiểm tra. FAILED hoặc INCOMPLETE được xếp lại hàng đợi chỉ cho các ID đó, tối đa 3 lượt đã đo cho mỗi ID. Sau lượt cuối, bảng vẫn ghi rõ lỗi; “đã chốt” bao gồm các lỗi đã dùng hết lượt. Crawler báo PDF_ONLY khi gặp ít nhất một version thuộc loại này. Các dấu .complete trong archive lưu trạng thái của từng version. Lỗi hạ tầng hoặc thiếu profiler không tiêu hao ba lượt của một ID. Nếu runtime gặp lỗi hạ tầng 3 lần liên tiếp, cell dừng và in nguyên nhân để sửa rồi chạy lại.

**Đọc thời gian.** *accepted_profiler_seconds* cộng thời gian profiler của các lượt hợp lệ, bao gồm các lần đo lại. *all_attempt_machine_seconds_observed* cộng thời gian quan sát của từng lượt, gồm phần lưu trữ và các lượt bị ngắt đến checkpoint cuối. *calendar_span_observed_s* là khoảng từ lượt đầu đến lần quan sát cuối, gồm cả lúc nhóm ngừng chạy. Các pha chạy song song có thể có tổng số giây lớn hơn wall time. Tổng byte và phân loại theo ID lấy lần đo đã ghi gần nhất. RSS trung bình dùng số mẫu làm trọng số, disk peak là đỉnh lấy mẫu trên từng VM. Profiler cũ có thể ghi rate_limit_sleep bằng 0 ở pipeline song song; số liệu ngủ/chờ đo bổ sung nằm trong network.json.gz và đã được tính trong wall time.

**Run all sau khi đủ 20.000 ID.** Notebook chỉ đọc kết quả và dựng bảng/biểu đồ. Muốn xem tiến độ trong lúc nhóm đang chạy, mở progress.md trên GitHub, hoặc chạy cell cuối sau khi đã chạy phần khởi tạo.

In [ ]:
#@title 7. Xem kết quả mới nhất từ repo
final_report = dashboard(JOURNAL, GROUPS, REPORT)